In [ ]:
import pandas as pd
import numpy as np
import re
import os
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer


# Loading Dataset and converting in DataFrame
dt = pd.read_csv('dataset.csv')
df=pd.DataFrame(dt)
print(df.head())
nltk.data.path.append(os.path.expanduser('~/nltk_data'))

# Download necessary NLTK resource
nltk.download('punkt',quiet=True)
nltk.download('stopwords',quiet=True)
nltk.download("wordnet",quiet=True)

TXT_COL= "sentence"
LABEL_COL = 'label'


# Text cleaning
def clean_text(text):
    if not isinstance(text,str):
        return ""

#     Lowercase
    text = text.lower()
    # Remove URLs
    text = re.sub(r"http\S+|www\S+", '',text)

    # Tokenize and clean
    tokens = re.findall(r'[a-z]+|\?',text)

    if not tokens:
        return ""

    stop_words = set(stopwords.words('english'))
    question_words = {'what','why','how','when','where','who'}
    stop_words -= question_words

    lemmatizer = WordNetLemmatizer()
    cleaned_tokens = [lemmatizer.lemmatize(tokens) for token in tokens if not  stop_words ]

    return " ".join(cleaned_tokens)

In [ ]:
df['cleaned_text'] = df[TXT_COL].apply(clean_text)

df['target']= df[LABEL_COL].apply(lambda x:1 if 'question' in str(x).lower() else 0)

print(df[['cleaned_text','target']].head(2))


In [ ]:

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

x = df['cleaned_text']
y = df['target']
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.2, random_state = 42)

print(f"training size: {len(x_train)}, testing size: {len(x_test)}")

In [ ]:
print("First 5 rows of X_train:")
print(x_train.head())
print("\nAre there any empty strings?")
print(x_train[x_train == ""].count())


In [ ]:


tfdif = TfidfVectorizer(max_features=500, ngram_range=(1,2),min_df=1,token_pattern=r'(?u)\b\w+\b\?',stop_words='english')
#
x_train_tfdif = tfdif.fit_transform(x_train)
x_test_tfdif = tfdif.tranform(x_test)
#
print(f"TF-IDF matrix shape: {x_train_tfdif.shape}")
